# Chapter 1 Cohort Characterization

This notebook characterizes the retained Chapter 1 cohort at the **stay** level and stratifies the summary by split (`train`, `validation`, `test`) plus an overall column.

It anchors on the canonical retained stay table in `artifacts/chapter1/cohort/` and joins the harmonized ASIC static table to recover age-group, sex, and hospital length-of-stay fields.

In [1]:
from __future__ import annotations

import os
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 200)


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    required_path = Path("artifacts/chapter1/cohort/chapter1_retained_stay_table.csv")
    for candidate in [start, *start.parents]:
        if (candidate / required_path).exists():
            return candidate
    raise FileNotFoundError(
        "Could not find the repository root from the current working directory. "
        "Expected to locate artifacts/chapter1/cohort/chapter1_retained_stay_table.csv."
    )


def find_static_harmonized_path(repo_root: Path) -> Path:
    candidates: list[Path] = []

    asic_input_root = os.environ.get("ASIC_INPUT_ROOT")
    if asic_input_root:
        candidates.append(Path(asic_input_root) / "static" / "harmonized.csv")

    candidates.extend(
        [
            repo_root / "artifacts" / "asic_harmonized" / "static" / "harmonized.csv",
            repo_root / "artifacts" / "asic_harmonized_full" / "static" / "harmonized.csv",
            repo_root.parent / "artifacts" / "asic_harmonized" / "static" / "harmonized.csv",
            repo_root.parent / "artifacts" / "asic_harmonized_full" / "static" / "harmonized.csv",
            repo_root.parent / "icu-data-platform" / "artifacts" / "asic_harmonized" / "static" / "harmonized.csv",
            repo_root.parent / "icu-data-platform" / "artifacts" / "asic_harmonized_full" / "static" / "harmonized.csv",
        ]
    )

    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    searched = "\n".join(f"- {candidate}" for candidate in candidates)
    raise FileNotFoundError(
        "Could not locate a harmonized ASIC static table for age/sex enrichment. "
        "Set ASIC_INPUT_ROOT or place the artifacts in one of these locations:\n"
        f"{searched}"
    )


repo_root = find_repo_root()
cohort_path = repo_root / "artifacts/chapter1/cohort/chapter1_retained_stay_table.csv"
split_path = repo_root / "artifacts/chapter1/splits/chapter1_stay_split_assignments.csv"
static_path = find_static_harmonized_path(repo_root)

input_manifest = pd.DataFrame(
    [
        {"artifact": "retained_cohort", "path": str(cohort_path)},
        {"artifact": "stay_split_assignments", "path": str(split_path)},
        {"artifact": "static_harmonized", "path": str(static_path)},
    ]
)

display(Markdown("## Inputs"))
display(input_manifest)

## Inputs

,artifact,path
0,retained_cohort,/Users/joanameyer/repository/1-mortality-decomposition/artifacts/chapter1/cohort/chapter1_retained_stay_table.csv
1,stay_split_assignments,/Users/joanameyer/repository/1-mortality-decomposition/artifacts/chapter1/splits/chapter1_stay_split_assignments.csv
2,static_harmonized,/Users/joanameyer/repository/icu-data-platform/artifacts/asic_harmonized/static/harmonized.csv


In [2]:
cohort = pd.read_csv(cohort_path)
splits = pd.read_csv(split_path)
static = pd.read_csv(
    static_path,
    usecols=["stay_id_global", "hospital_id", "age_group", "sex", "hosp_los"],
)

for frame in (cohort, splits, static):
    frame["stay_id_global"] = frame["stay_id_global"].astype("string")
    frame["hospital_id"] = frame["hospital_id"].astype("string")

splits["split"] = pd.Categorical(
    splits["split"],
    categories=["train", "validation", "test"],
    ordered=True,
)

analytic = (
    cohort.merge(
        splits[["stay_id_global", "split"]],
        on="stay_id_global",
        how="left",
        validate="one_to_one",
    )
    .merge(
        static,
        on=["stay_id_global", "hospital_id"],
        how="left",
        validate="one_to_one",
    )
    .sort_values(["split", "hospital_id", "stay_id_global"], na_position="last")
    .reset_index(drop=True)
)

analytic["icu_mortality"] = pd.to_numeric(analytic["icu_mortality"], errors="coerce")
analytic["readmission"] = pd.to_numeric(analytic["readmission"], errors="coerce")
analytic["hosp_los"] = pd.to_numeric(analytic["hosp_los"], errors="coerce")
analytic["icu_end_time_proxy_hours"] = pd.to_numeric(
    analytic["icu_end_time_proxy_hours"],
    errors="coerce",
)
analytic["icu_proxy_los_days"] = analytic["icu_end_time_proxy_hours"] / 24.0

if analytic["stay_id_global"].duplicated().any():
    duplicate_ids = analytic.loc[
        analytic["stay_id_global"].duplicated(keep=False),
        "stay_id_global",
    ].drop_duplicates().tolist()
    raise ValueError(f"Merged analytic table has duplicate stay IDs: {duplicate_ids[:10]}")

if analytic["split"].isna().any():
    missing_split_ids = analytic.loc[analytic["split"].isna(), "stay_id_global"].tolist()
    raise ValueError(f"Missing split assignments for retained stays: {missing_split_ids[:10]}")

audit = pd.DataFrame(
    [
        {"metric": "retained_stays", "value": int(analytic["stay_id_global"].nunique())},
        {"metric": "retained_hospitals", "value": int(analytic["hospital_id"].nunique())},
        {"metric": "missing_age_group", "value": int(analytic["age_group"].isna().sum())},
        {"metric": "missing_sex", "value": int(analytic["sex"].isna().sum())},
        {"metric": "missing_hospital_los", "value": int(analytic["hosp_los"].isna().sum())},
        {"metric": "missing_icu_proxy_los", "value": int(analytic["icu_proxy_los_days"].isna().sum())},
    ]
)

display(Markdown("## Analytic dataset audit"))
display(audit)
display(analytic.head())

## Analytic dataset audit

,metric,value
0,retained_stays,35
1,retained_hospitals,4
2,missing_age_group,0
3,missing_sex,0
4,missing_hospital_los,7
5,missing_icu_proxy_los,0


,stay_id_global,hospital_id,readmission,icu_mortality,icd10_codes,icu_admission_time,icu_end_time_proxy,icu_end_time_proxy_hours,has_dynamic_data,mech_vent_ge_24h_qc,adult_age_ge_18_assumed_upstream,first_stay_proxy_eligible,split,age_group,sex,hosp_los,icu_proxy_los_days
0,asic_UK02_9991,asic_UK02,0.0,0.0,"R77,U69,Z85,N39,I10,C91,A49,J15,I48,T84,Z96,J96,B95,D68,D62,D69,R65,M00,A41,T84,Z96,B95",0,20 days 16:30:00,496.50,True,True,True,True,train,70-79,M,35.0,20.687500
1,asic_UK02_9992,asic_UK02,0.0,1.0,"S27,E03,J96,M17,J98,U99,J93,I50,U69,G20,Z11,Z85,E11,I46",0,1 days 12:15:00,36.25,True,True,True,True,train,<70,M,1.0,1.510417
2,asic_UK02_9993,asic_UK02,0.0,0.0,"C09,E03,Z93,I10,E87,Z92,I25,Z43,C10",0,4 days 23:30:00,119.50,True,True,True,True,train,70-79,M,15.0,4.979167
3,asic_UK02_9994,asic_UK02,0.0,1.0,"C34,J96,K86,J44,I10,I11,G40",0,35 days 09:15:00,849.25,True,True,True,True,train,70-79,F,10.0,35.385417
4,asic_UK02_9996,asic_UK02,0.0,0.0,"G93,I64,J18,E23,I60,U69,I61,I67",0,2 days 20:45:00,68.75,True,True,True,True,train,70-79,M,45.0,2.864583


In [3]:
SPLIT_SEQUENCE = [
    ("Overall", None),
    ("Train", "train"),
    ("Validation", "validation"),
    ("Test", "test"),
]


def iter_split_frames(df: pd.DataFrame):
    for label, split_name in SPLIT_SEQUENCE:
        if split_name is None:
            yield label, df
        else:
            yield label, df[df["split"] == split_name]


def format_integer(value: int | float) -> str:
    return f"{int(value):,}"


def format_count_pct(count: int | float, denominator: int | float) -> str:
    if denominator in (0, None) or pd.isna(denominator):
        return "NA"
    return f"{int(count)} ({count / denominator:.1%})"


def format_median_iqr(series: pd.Series) -> str:
    numeric = pd.to_numeric(series, errors="coerce").dropna()
    if numeric.empty:
        return "NA"
    return (
        f"{numeric.median():.1f} "
        f"[{numeric.quantile(0.25):.1f}, {numeric.quantile(0.75):.1f}]"
    )


def ordered_levels(series: pd.Series, preferred_order: list[str]) -> list[str]:
    observed = set(series.dropna().astype(str))
    ordered = [level for level in preferred_order if level in observed]
    extras = sorted(observed - set(ordered))
    return ordered + extras


age_group_levels = ordered_levels(analytic["age_group"], ["<70", "70-79", "80-130"])
sex_levels = ordered_levels(analytic["sex"], ["F", "M"])


def build_table1(df: pd.DataFrame) -> pd.DataFrame:
    frames = dict(iter_split_frames(df))
    rows: list[dict[str, str]] = []

    def append_row(group: str, statistic: str, formatter) -> None:
        row = {"Group": group, "Statistic": statistic}
        for label, frame in frames.items():
            row[label] = formatter(frame)
        rows.append(row)

    append_row(
        "Cohort",
        "Stays, n",
        lambda frame: format_integer(frame["stay_id_global"].nunique()),
    )
    append_row(
        "Cohort",
        "Hospitals, n",
        lambda frame: format_integer(frame["hospital_id"].nunique()),
    )
    append_row(
        "Outcome",
        "In-ICU mortality, n (%)",
        lambda frame: format_count_pct(
            frame["icu_mortality"].eq(1).sum(),
            frame["stay_id_global"].nunique(),
        ),
    )
    append_row(
        "Outcome",
        "Readmission, n (%)",
        lambda frame: format_count_pct(
            frame["readmission"].eq(1).sum(),
            frame["stay_id_global"].nunique(),
        ),
    )
    append_row(
        "LOS",
        "ICU end-time proxy LOS, median [Q1, Q3] days",
        lambda frame: format_median_iqr(frame["icu_proxy_los_days"]),
    )
    append_row(
        "LOS",
        "Hospital LOS available, n (%)",
        lambda frame: format_count_pct(
            frame["hosp_los"].notna().sum(),
            frame["stay_id_global"].nunique(),
        ),
    )
    append_row(
        "LOS",
        "Hospital LOS, median [Q1, Q3] days",
        lambda frame: format_median_iqr(frame["hosp_los"]),
    )

    for age_group in age_group_levels:
        append_row(
            "Age group",
            f"{age_group}, n (%)",
            lambda frame, age_group=age_group: format_count_pct(
                frame["age_group"].astype(str).eq(age_group).sum(),
                frame["stay_id_global"].nunique(),
            ),
        )

    if df["age_group"].isna().any():
        append_row(
            "Age group",
            "Missing, n (%)",
            lambda frame: format_count_pct(
                frame["age_group"].isna().sum(),
                frame["stay_id_global"].nunique(),
            ),
        )

    for sex in sex_levels:
        append_row(
            "Sex",
            f"{sex}, n (%)",
            lambda frame, sex=sex: format_count_pct(
                frame["sex"].astype(str).eq(sex).sum(),
                frame["stay_id_global"].nunique(),
            ),
        )

    if df["sex"].isna().any():
        append_row(
            "Sex",
            "Missing, n (%)",
            lambda frame: format_count_pct(
                frame["sex"].isna().sum(),
                frame["stay_id_global"].nunique(),
            ),
        )

    return pd.DataFrame(rows).set_index(["Group", "Statistic"])


def build_hospital_summary(df: pd.DataFrame) -> pd.DataFrame:
    total_stays = df["stay_id_global"].nunique()
    rows: list[dict[str, object]] = []

    for hospital_id in sorted(df["hospital_id"].dropna().astype(str).unique().tolist()):
        hospital_df = df[df["hospital_id"].astype(str) == hospital_id]
        row = {
            "Hospital": hospital_id,
            "Overall stays": int(hospital_df["stay_id_global"].nunique()),
            "Overall share": f"{hospital_df['stay_id_global'].nunique() / total_stays:.1%}",
            "Train stays": int(hospital_df[hospital_df["split"] == "train"]["stay_id_global"].nunique()),
            "Validation stays": int(hospital_df[hospital_df["split"] == "validation"]["stay_id_global"].nunique()),
            "Test stays": int(hospital_df[hospital_df["split"] == "test"]["stay_id_global"].nunique()),
            "In-ICU mortality": format_count_pct(
                hospital_df["icu_mortality"].eq(1).sum(),
                hospital_df["stay_id_global"].nunique(),
            ),
            "ICU end-time proxy LOS, median [Q1, Q3] days": format_median_iqr(
                hospital_df["icu_proxy_los_days"]
            ),
            "Hospital LOS, median [Q1, Q3] days": format_median_iqr(hospital_df["hosp_los"]),
        }
        rows.append(row)

    return pd.DataFrame(rows)


In [4]:
display(Markdown("## Table 1: Overall and split-specific stay characteristics"))
display(Markdown("Unit of analysis: retained Chapter 1 ICU stays."))

table1 = build_table1(analytic)
display(table1)

## Table 1: Overall and split-specific stay characteristics

Unit of analysis: retained Chapter 1 ICU stays.

Overall  \
Group     Statistic                                                         
Cohort    Stays, n                                                     35   
          Hospitals, n                                                  4   
Outcome   In-ICU mortality, n (%)                              10 (28.6%)   
          Readmission, n (%)                                     0 (0.0%)   
LOS       ICU end-time proxy LOS, median [Q1, Q3] days   11.7 [4.8, 19.6]   
          Hospital LOS available, n (%)                        28 (80.0%)   
          Hospital LOS, median [Q1, Q3] days            19.0 [12.2, 28.2]   
Age group <70, n (%)                                           18 (51.4%)   
          70-79, n (%)                                         15 (42.9%)   
          80-130, n (%)                                          2 (5.7%)   
Sex       F, n (%)                                             12 (34.3%)   
          M, n (%)                                             23 (65.7%)   

                                                                    Train  \
Group     Statistic                                                         
Cohort    Stays, n                                                     25   
          Hospitals, n                                                  4   
Outcome   In-ICU mortality, n (%)                               8 (32.0%)   
          Readmission, n (%)                                     0 (0.0%)   
LOS       ICU end-time proxy LOS, median [Q1, Q3] days    7.6 [4.0, 20.7]   
          Hospital LOS available, n (%)                        20 (80.0%)   
          Hospital LOS, median [Q1, Q3] days            17.5 [12.2, 24.2]   
Age group <70, n (%)                                           10 (40.0%)   
          70-79, n (%)                                         13 (52.0%)   
          80-130, n (%)                                          2 (8.0%)   
Sex       F, n (%)                                              9 (36.0%)   
          M, n (%)                                             16 (64.0%)   

                                                               Validation  \
Group     Statistic                                                         
Cohort    Stays, n                                                      6   
          Hospitals, n                                                  4   
Outcome   In-ICU mortality, n (%)                               2 (33.3%)   
          Readmission, n (%)                                     0 (0.0%)   
LOS       ICU end-time proxy LOS, median [Q1, Q3] days   10.6 [9.3, 15.3]   
          Hospital LOS available, n (%)                         5 (83.3%)   
          Hospital LOS, median [Q1, Q3] days            18.0 [18.0, 22.0]   
Age group <70, n (%)                                            5 (83.3%)   
          70-79, n (%)                                          1 (16.7%)   
          80-130, n (%)                                          0 (0.0%)   
Sex       F, n (%)                                              2 (33.3%)   
          M, n (%)                                              4 (66.7%)   

                                                                     Test  
Group     Statistic                                                        
Cohort    Stays, n                                                      4  
          Hospitals, n                                                  4  
Outcome   In-ICU mortality, n (%)                                0 (0.0%)  
          Readmission, n (%)                                     0 (0.0%)  
LOS       ICU end-time proxy LOS, median [Q1, Q3] days  15.3 [14.2, 21.1]  
          Hospital LOS available, n (%)                         3 (75.0%)  
          Hospital LOS, median [Q1, Q3] days            35.0 [22.0, 77.5]  
Age group <70, n (%)                                            3 (75.0%)  
          70-79, n (%)                                          1 (25

In [5]:
display(Markdown("## Hospital-level summary"))

hospital_summary = build_hospital_summary(analytic)
display(hospital_summary)

## Hospital-level summary

,Hospital,Overall stays,Overall share,Train stays,Validation stays,Test stays,In-ICU mortality,"ICU end-time proxy LOS, median [Q1, Q3] days","Hospital LOS, median [Q1, Q3] days"
0,asic_UK02,10,28.6%,7,2,1,4 (40.0%),"11.8 [5.7, 15.6]","20.5 [15.0, 35.0]"
1,asic_UK04,9,25.7%,7,1,1,2 (22.2%),"9.6 [6.9, 14.5]","10.0 [9.0, 15.0]"
2,asic_UK07,9,25.7%,6,2,1,3 (33.3%),"9.3 [5.0, 36.9]","23.0 [21.0, 29.0]"
3,asic_UK08,7,20.0%,5,1,1,1 (14.3%),"16.0 [3.3, 42.2]",NA
